# Data Preprocessing — Qué Cocinar IA

Este notebook carga `data/enriched_recipes.csv`, limpia los datos y los indexa en ChromaDB.

In [ ]:
import ast
import re
from pathlib import Path

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

PROJECT_ROOT = Path("..").resolve()
CSV_PATH = PROJECT_ROOT / "data" / "enriched_recipes.csv"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
COLLECTION_NAME = "enriched_recipes"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 500

In [ ]:
# Load local recipes dataset
if not CSV_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"Archivo: {CSV_PATH}")
print(f"Filas: {len(df):,} | Columnas: {list(df.columns)}")
df.head(2)

In [ ]:
def row_to_document(row) -> Document | None:
    """Convert a DataFrame row into a LangChain Document with scalar metadata."""
    name = str(row["recipe_name"]).strip()
    ingredients = str(row["ingredients"]).strip()
    meal_type = str(row["meal_type"]).strip()
    taste_profile = str(row["taste_profile"]).strip()
    served_temperature = str(row["served_temperature"]).strip()
    season = str(row["season"]).strip()
    difficulty = str(row["difficulty"]).strip()
    characteristics = str(row["characteristics"]).strip()
    main_ingredients = str(row["main_ingredients"]).strip()

    page_content = (
        f"Name: {name}\n\n"
        f"Meal type_: {meal_type}\n\n"
        f"Taste profile: {taste_profile}\n\n"
        f"Season: {season}\n\n"
        f"Difficulty: {difficulty}\n\n"
        f"Main Ingredients: {main_ingredients}\n\n"
        f"Characteristics: {characteristics}\n\n"
        f"Ingredients:\n{ingredients}"
    )
    
    metadata = {
        "recipe_id": int(row["recipe_id"])
    }

    # Chroma only accepts str, int, float, bool — drop None values
    metadata = {k: v for k, v in metadata.items() if v is not None}
    return Document(page_content=page_content, metadata=metadata)

documents = [doc for doc in (row_to_document(row) for _, row in df.iterrows()) if doc]
print(f"Documentos listos: {len(documents):,}")
print(documents[0].page_content[:300], "...\n")
print("Metadata ejemplo:", documents[0].metadata)

In [ ]:
# Embed and persist to ChromaDB (batched to limit memory)
# IMPORTANT: delete old index first so recipe_id is present on every document
import shutil

if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
    print(f"Índice anterior eliminado: {CHROMA_DIR}")

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

vectorstore = None
for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i : i + BATCH_SIZE]
    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(CHROMA_DIR),
        )
    else:
        vectorstore.add_documents(batch)
    print(f"Indexados {min(i + BATCH_SIZE, len(documents)):,} / {len(documents):,}")

assert "csv_row_id" in documents[0].metadata
print(f"\nChromaDB persistido en: {CHROMA_DIR}")
print("Metadata ejemplo:", vectorstore.similarity_search("chicken", k=1)[0].metadata)